# 1. Dataset

In [1]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

X, y = make_regression(n_samples=200, n_features=2, noise=10, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("X Shape:", X.shape)
print("y Shape:", y.shape)

X Shape: (200, 2)
y Shape: (200,)


# 2. Sklearn 

In [2]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

sk_model = GradientBoostingRegressor(n_estimators=10, learning_rate=0.1, max_depth=1, random_state=42)

sk_model.fit(X_train, y_train)

sk_pred = sk_model.predict(X_test)

print("Sklearn MSE:", mean_squared_error(y_test, sk_pred))
print("Sklearn R2:", r2_score(y_test, sk_pred))

Sklearn MSE: 1672.4745067537835
Sklearn R2: 0.4266800942355421


# 3. Scratch Code

In [3]:
from sklearn.tree import DecisionTreeRegressor

class GradientBoostingScratch:

    def __init__(self, n_estimators=10, lr=0.1):
        self.n_estimators = n_estimators
        self.lr = lr
        self.models = []

    def fit(self, X, y):

        self.initial = np.mean(y)
        pred = np.full(len(y), self.initial)

        for _ in range(self.n_estimators):

            residual = y - pred

            model = DecisionTreeRegressor(max_depth=1)
            model.fit(X, residual)

            update = model.predict(X)
            pred += self.lr * update

            self.models.append(model)

    def predict(self, X):

        pred = np.full(len(X), self.initial)

        for model in self.models:
            pred += self.lr * model.predict(X)

        return pred

In [4]:
my_model = GradientBoostingScratch(n_estimators=10, lr=0.1)

my_model.fit(X_train, y_train)

my_pred = my_model.predict(X_test)

print("Scratch MSE:", mean_squared_error(y_test, my_pred))
print("Scratch R2:", r2_score(y_test, my_pred))

Scratch MSE: 1672.4745067537835
Scratch R2: 0.4266800942355421


In [5]:
print("Sklearn MSE:", mean_squared_error(y_test, sk_pred))
print("Scratch MSE:", mean_squared_error(y_test, my_pred))

print("Sklearn R2:", r2_score(y_test, sk_pred))
print("Scratch R2:", r2_score(y_test, my_pred))

Sklearn MSE: 1672.4745067537835
Scratch MSE: 1672.4745067537835
Sklearn R2: 0.4266800942355421
Scratch R2: 0.4266800942355421


# 4. Gradient Boosting — Important Formulas

## 1. Initial Prediction

For regression, start with the mean:

$$
F_0(x)=\frac{1}{n}\sum_{i=1}^{n}y_i
$$

---

## 2. Residual

For MSE loss:

$$
r_i=y_i-F_{m-1}(x_i)
$$

The next weak learner is trained to predict these residuals.

---

## 3. Weak Learner

$$
h_m(x)\approx r
$$

where $h_m$ is usually a small decision tree.

---

## 4. Model Update

$$
F_m(x)
=
F_{m-1}(x)
+
\eta h_m(x)
$$

where $\eta$ is the learning rate.

---

## 5. Final Prediction

After $M$ trees:

$$
F_M(x)
=
F_0(x)
+
\eta\sum_{m=1}^{M}h_m(x)
$$

---

## 6. MSE Loss

$$
L
=
\frac{1}{n}
\sum_{i=1}^{n}
(y_i-\hat y_i)^2
$$

---

## 7. Gradient

For MSE:

$$
\frac{\partial L}{\partial \hat y_i}
=
\frac{2}{n}(\hat y_i-y_i)
$$

The negative gradient is proportional to the residual:

$$
-\frac{\partial L}{\partial \hat y_i}
\propto
y_i-\hat y_i
$$

Therefore, the next tree learns the residuals.

---

## 8. Main Idea

$$
\boxed{
\text{Previous Prediction}
\rightarrow
\text{Residual}
\rightarrow
\text{New Tree}
\rightarrow
\text{Update}
}
$$